# SPS-CA — Optimized 1000-Test Evolution Proof

Minimal Colab notebook: setup, one-cell 1000-case execution, evidence inspection, and optional full pytest.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
REPO = Path('/content/SPS_CA')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Repo:', REPO)

In [ ]:
# Install only the declared Python dependencies (Colab already provides Python/pytest).
!pip install -q -r requirements.txt

In [ ]:
# ONE CELL — execute all 1000 growth/evolution tests
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'testing/test_sps_scenarios.py'], cwd=REPO, text=True)
print(result.stdout)
print(result.stderr)
raise SystemExit(result.returncode)

In [ ]:
# Evidence summary. Live runtime evidence is reported if present.
paths = [REPO/'runtime/evolution_events.json', REPO/'capabilities/registry.json']
for p in paths:
    print(f'\n--- {p.relative_to(REPO)} ---')
    if not p.exists():
        print('not present')
        continue
    data = json.loads(p.read_text(encoding='utf-8'))
    if isinstance(data, list):
        print('events:', len(data))
    elif isinstance(data, dict) and 'capabilities' in data:
        caps = data['capabilities']
        print('registered:', len(caps))
        print('generated:', sum(1 for c in caps if c.get('generated')))
        print('total reuses:', sum(int(c.get('reuse_count', 0) or 0) for c in caps))
        for c in caps:
            if c.get('generated'):
                print('generated capability:', c.get('id'), 'reuse_count=', c.get('reuse_count', 0))

In [ ]:
# Optional: run every repository pytest after the focused 1000-case suite.
# subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO, check=True)